# M14 L14: AI-Native Data Platforms -- Lakehouse, Vector DBs, Contracts

**Build a tiny lakehouse, vector index, feature store, data contract, and lineage graph from scratch.**

Duration: 90 min.  No Spark, no Iceberg, no cloud account.

## Plain English (the 12-year-old version)

A modern data platform has **five things** that have to talk to each other:

1. A **lakehouse** (cheap storage with ACID, snapshots, time travel)
2. A **vector index** (a peer to OLTP/OLAP, queryable like a SQL table)
3. A **feature store** (online + offline, with point-in-time correctness)
4. A **data contract** (owner, SLA, schema version -- the interface)
5. A **lineage graph** (DAG of tables + transforms, topologically sortable)

The "AI-native" part is that the vector index sits beside OLTP/OLAP as a peer, not behind a sidecar microservice. And ELT (load raw first, transform in SQL) beats ETL (transform before load) because storage is cheap and warehouses are fast.

## The 5 things to know

1. **Lakehouse = open table format on cheap storage** -- Iceberg / Delta / Hudi give you ACID, snapshots, time travel at lake economics.
2. **Vector index is a peer data product** -- not a microservice. You can `SELECT ... WHERE kth_nearest(...) <= 5` in the same SQL.
3. **Point-in-time correctness** -- training labels must only see features known at the label timestamp; the latest features leak future info.
4. **Data contract** -- owner + SLA + schema version. Without it, schemas rot silently. With it, breaking changes are caught at publish time.
5. **Lineage + impact analysis** -- `impact("orders")` answers "what breaks if this table changes?" in milliseconds.

In [ ]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm14-data-platforms' / 'L14-data-platforms.py'
_spec = importlib.util.spec_from_file_location('L14', _p)
_L14 = importlib.util.module_from_spec(_spec)
sys.modules['L14'] = _L14
_spec.loader.exec_module(_L14)

print(f'Setup OK. Lesson module: {_L14.LESSON_NUMBER}')

In [ ]:
# Run the 15 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L14, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

In [ ]:
# === Section 1: Table + Column schemas (the data contract primitive) ===
users = _L14.make_users_table()
orders = _L14.make_orders_table()
products = _L14.make_products_table()
print(f'users:    {users}')
print(f'orders:   {orders}')
print(f'products: {products}')
print()
# Filter + select -- the SQL-style operations
us_orders = orders.filter(lambda r: r["amount"] > 30).select("order_id", "amount")
print(f'orders > 30: {[r for r in us_orders.rows]}')

In [ ]:
# === Section 2: DataContract (owner + SLA + version) ===
contract = _L14.DataContract(
    table=users, owner='growth-team',
    sla_freshness_min=30, schema_version=1,
    description='Core users table -- source of truth for all user features.',
)
contract.add_consumer('ml-team')
contract.add_consumer('marketing-team')
print(f'owner:     {contract.owner}')
print(f'SLA:       {contract.sla_freshness_min} min')
print(f'version:   {contract.schema_version}')
print(f'consumers: {contract.consumers}')

In [ ]:
# === Section 3: LakehouseTable (Iceberg/Delta style) ===
lh = _L14.LakehouseTable(
    name='orders_lh',
    columns=[
        _L14.Column('order_id', 'int', nullable=False),
        _L14.Column('user_id', 'int', nullable=False),
        _L14.Column('amount', 'float', nullable=False),
    ],
    partition_by=['user_id'],
)
lh.append([{'order_id': 1, 'user_id': 1, 'amount': 10.0}, {'order_id': 2, 'user_id': 2, 'amount': 20.0}])
lh.append([{'order_id': 3, 'user_id': 1, 'amount': 30.0}])
print(f'snapshots:  {len(lh.snapshots)}')
print(f'latest:     version={lh.latest().version}, rows={len(lh)}')
print(f'v1 rows:    {len(lh.at_version(1).rows)}')
print(f'v2 rows:    {len(lh.at_version(2).rows)}')
print(f'partitions: {lh.partitions()}')

In [ ]:
# === Section 4: FeatureStore (online + point-in-time) ===
fs = _L14.FeatureStore()
view = _L14.FeatureView(name='user_orders_30d', version=1, features=['orders_30d', 'gmv_30d'])
view.rows = [
    _L14.FeatureRow('u1', {'orders_30d': 1, 'gmv_30d': 29.99}, '2024-02-01T00:00:00'),
    _L14.FeatureRow('u1', {'orders_30d': 2, 'gmv_30d': 79.98}, '2024-02-15T00:00:00'),
    _L14.FeatureRow('u2', {'orders_30d': 1, 'gmv_30d': 49.99}, '2024-02-20T00:00:00'),
]
fs.register(view)
# Online (latest)
print(f'online u1:           {fs.get_online("user_orders_30d", "u1")}')
# Point-in-time (must use the feature row <= label_ts, not the latest)
print(f'pit u1 @ 2024-02-10: {fs.point_in_time("user_orders_30d", "u1", "2024-02-10T00:00:00")}')
print(f'pit u1 @ 2024-03-01: {fs.point_in_time("user_orders_30d", "u1", "2024-03-01T00:00:00")}')

In [ ]:
# === Section 5: VectorIndex (peer to OLTP/OLAP) ===
idx = _L14.make_embeddings_index()
q = [0.85, 0.15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
hits = idx.query(q, top_k=3)
print(f'top-3 nearest to q:')
for row, score in hits:
    print(f'  {row.id}  score={score:.4f}  payload={row.payload}')
# SQL-like filter on payload
hardware = idx.select(where=lambda r: r.payload['product_id'] in (1001, 1002))
print(f'\npayload-selected (hardware): {[r.id for r in hardware]}')

In [ ]:
# === Section 6: Pipeline (ELT) ===
products_lh = _L14.LakehouseTable(
    name='products_lh',
    columns=[
        _L14.Column('product_id', 'int', nullable=False),
        _L14.Column('name', 'str', nullable=False),
        _L14.Column('price', 'float', nullable=False),
    ],
)
def clean_products(rows):
    out = []
    for r in rows:
        r2 = {k: v for k, v in r.items() if k in ('product_id', 'name', 'price')}
        r2['name'] = r2['name'].strip().title()
        out.append(r2)
    return out
pipe = _L14.Pipeline(name='products_elt', source_name='postgres.products', target=products_lh, transform=clean_products)
raw_source = [
    {'product_id': 1001, 'name': '  widget ', 'price': 29.99, 'secret_internal_col': 'x'},
    {'product_id': 1002, 'name': 'gadget', 'price': 49.99, 'secret_internal_col': 'y'},
]
pipe.run(raw_source)
print(f'pipeline runs:  {pipe.runs}')
print(f'latest rows:    {products_lh.latest().rows}')

In [ ]:
# === Section 7: LineageGraph (DAG + topological sort + impact) ===
g = _L14.LineageGraph()
for nid, kind, lbl in [
    ('src_pg', 'source', 'postgres'),
    ('raw_orders', 'table', 'orders (raw)'),
    ('clean_orders', 'table', 'orders (clean)'),
    ('agg_user_orders', 'table', 'user_orders_30d'),
    ('ml_features', 'sink', 'ML training set'),
]:
    g.add_node(_L14.LineageNode(nid, kind, lbl))
g.add_edge('src_pg', 'raw_orders', 'snapshot')
g.add_edge('raw_orders', 'clean_orders', 'drop nulls + dedupe')
g.add_edge('clean_orders', 'agg_user_orders', 'group by user_id, 30d window')
g.add_edge('agg_user_orders', 'ml_features', 'join labels')
order = g.topological_sort()
print(f'execution order:           {order}')
print(f'impact of raw_orders:      {g.impact("raw_orders")}')
print(f'impact of clean_orders:    {g.impact("clean_orders")}')

## Architecture (Mermaid)

```mermaid
flowchart LR
    SRC[Sources<br/>Postgres / Kafka / S3] --> P[Pipeline<br/>ELT]
    P --> RAW[(Lakehouse<br/>raw / bronze)]
    RAW --> CLEAN[(Lakehouse<br/>clean / silver)]
    CLEAN --> AGG[(Lakehouse<br/>aggregated / gold)]
    RAW --> VEC[VectorIndex<br/>peer to OLAP]
    CLEAN --> FS_OFF[FeatureStore<br/>offline]
    FS_OFF --> FS_ON[FeatureStore<br/>online<br/>Redis / DynamoDB]
    AGG --> ML[ML training set]
    FS_ON --> ML
    VEC --> APP[RAG app]
    AGG --> BI[BI dashboards]
    DC[DataContract<br/>owner / SLA / version] -. enforces .-> RAW
    DC -. enforces .-> CLEAN
    LG[LineageGraph] -. tracks .-> RAW
    LG -. tracks .-> CLEAN
    LG -. tracks .-> AGG
```

**Read as:** sources -> raw -> clean -> aggregated, with feature store, vector index, data contract, and lineage graph layered on top. Every table has an owner and an SLA; every transform is in the lineage.

## Reflect (10 min)

1. Why is ELT (extract -> load -> transform) better than ETL (extract -> transform -> load) in a lakehouse world?
2. What does "point-in-time correctness" mean, and what goes wrong if you ignore it?
3. Why is a vector index a "peer" to OLTP/OLAP rather than a sidecar microservice? What does that buy you?
4. Walk through the lineage graph: which tables break if `raw_orders` schema changes? How do you know?
5. (Stretch) Add a `Snapshot.expire(self, keep_last_n)` method to the lakehouse that drops the oldest snapshots.

---

**Read next:**
- `lessons/m15-staff-design/` -- staff-level system design
- `lessons/m16-fullstack/` -- fullstack AI applications
- `SYLLABUS.md` -- the 18 modules and 21-day plan